# EduPath Coach: 맞춤형 학습 코치 에이전트

교육/학습 테마를 위해 설계한 에이전트입니다. 학습자의 목표, 현재 수준, 사용 가능한 시간을 바탕으로 학습 계획을 만들고, 핵심 개념을 짧게 설명한 뒤, 바로 연습 문제까지 제공하는 흐름으로 구성했습니다.

## Step 1. 에이전트 설계

**이름**: EduPath Coach

**목적**: 학습자가 막연한 목표만 가지고 있어도, 현재 수준에 맞는 학습 경로와 즉시 실행 가능한 연습 문제를 제공해 학습 시작 장벽을 낮춥니다.

**핵심 기능**

1. 학습자 진단: 목표, 수준, 공부 가능 시간을 바탕으로 학습 격차를 추정합니다.
2. 학습 경로 생성: 오늘 바로 시작할 수 있는 단계별 학습 계획을 만듭니다.
3. 마이크로 튜터링: 핵심 개념을 짧고 이해하기 쉽게 설명합니다.
4. 퀴즈 생성: 설명 직후 확인 문제를 만들어 즉시 복습할 수 있게 합니다.
5. 피드백 정리: 다음 공부 액션을 한 문장으로 요약합니다.

**그래프 구조**

```mermaid
flowchart LR
    START([START]) --> Diagnose[diagnose_learner]
    Diagnose --> Teach[create_mini_lesson]
    Teach --> Quiz[build_quiz]
    Quiz --> Wrap[wrap_up]
    Wrap --> END([END])
```

**노드 설명**

- `diagnose_learner`: 학습자의 목표와 수준을 분석하고 학습 공백과 계획을 만듭니다.
- `create_mini_lesson`: 오늘 배워야 할 핵심을 짧은 미니 레슨으로 정리합니다.
- `build_quiz`: 미니 레슨을 바탕으로 확인 문제를 생성합니다.
- `wrap_up`: 다음 학습 행동을 간단히 요약합니다.

## Step 2. LangGraph 기초 구축

아래 코드는 커스텀 `State`를 정의하고, 최소 2개 이상의 작동하는 노드를 구현한 뒤, 기본 그래프를 연결하는 예시입니다. 외부 LLM 없이도 실행되도록 작성해서 구조 확인이 쉽습니다.

In [1]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


In [2]:
class LearningState(TypedDict):
    topic: str
    goal: str
    learner_level: str
    available_minutes: int
    knowledge_gaps: list[str]
    study_plan: list[str]
    lesson_summary: str
    quiz: list[dict[str, str]]
    final_feedback: str


def diagnose_learner(state: LearningState) -> LearningState:
    level = state["learner_level"].lower()
    topic = state["topic"]
    minutes = state["available_minutes"]

    if "beginner" in level or "초급" in level:
        gaps = [
            f"{topic}의 핵심 용어 이해",
            f"{topic}의 가장 기본적인 개념 구분",
            "짧은 예제를 보고 스스로 설명하는 연습",
        ]
    else:
        gaps = [
            f"{topic}의 개념을 실제 문제에 적용하는 연습",
            "설명은 알지만 손으로 구현하거나 표현하는 단계",
            "틀린 이유를 스스로 복기하는 습관",
        ]

    plan = [
        f"5분: {topic}에서 오늘 꼭 알아야 할 핵심 개념 3개 훑기",
        f"{max(minutes - 15, 10)}분: 예시 2개를 보며 개념을 직접 설명해 보기",
        "10분: 퀴즈를 풀고 틀린 이유를 한 줄로 정리하기",
    ]

    return {
        **state,
        "knowledge_gaps": gaps,
        "study_plan": plan,
    }


def create_mini_lesson(state: LearningState) -> LearningState:
    topic = state["topic"]
    goal = state["goal"]
    lesson = (
        f"오늘의 주제는 {topic}입니다. 목표는 '{goal}' 달성입니다. "
        f"먼저 {topic}의 정의를 한 문장으로 말할 수 있어야 하고, "
        f"그다음에는 대표 예시를 통해 언제 쓰는지 구분할 수 있어야 합니다. "
        f"마지막으로 배운 내용을 자신의 말로 다시 설명하면 학습이 더 오래 유지됩니다."
    )

    return {
        **state,
        "lesson_summary": lesson,
    }


def build_quiz(state: LearningState) -> LearningState:
    topic = state["topic"]
    quiz = [
        {
            "question": f"{topic}의 핵심 개념을 한 문장으로 설명해 보세요.",
            "type": "short_answer",
        },
        {
            "question": f"{topic}가 실제로 사용되는 상황을 한 가지 써 보세요.",
            "type": "application",
        },
        {
            "question": "오늘 배운 내용 중 아직 헷갈리는 부분을 한 가지 적어 보세요.",
            "type": "reflection",
        },
    ]

    return {
        **state,
        "quiz": quiz,
    }


def wrap_up(state: LearningState) -> LearningState:
    feedback = (
        f"다음 행동: {state['study_plan'][0]} 후, 퀴즈 3문항을 풀고 "
        f"헷갈린 개념은 '{state['knowledge_gaps'][0]}' 중심으로 다시 복습하세요."
    )

    return {
        **state,
        "final_feedback": feedback,
    }


In [3]:
builder = StateGraph(LearningState)

builder.add_node("diagnose_learner", diagnose_learner)
builder.add_node("create_mini_lesson", create_mini_lesson)
builder.add_node("build_quiz", build_quiz)
builder.add_node("wrap_up", wrap_up)

builder.add_edge(START, "diagnose_learner")
builder.add_edge("diagnose_learner", "create_mini_lesson")
builder.add_edge("create_mini_lesson", "build_quiz")
builder.add_edge("build_quiz", "wrap_up")
builder.add_edge("wrap_up", END)

graph = builder.compile()


In [6]:
sample_input = {
    "topic": "Python 함수",
    "goal": "함수를 직접 정의하고 호출할 수 있다",
    "learner_level": "beginner",
    "available_minutes": 30,
    "knowledge_gaps": [],
    "study_plan": [],
    "lesson_summary": "",
    "quiz": [],
    "final_feedback": "",
}

result = graph.invoke(sample_input)
result


{'topic': 'Python 함수',
 'goal': '함수를 직접 정의하고 호출할 수 있다',
 'learner_level': 'beginner',
 'available_minutes': 30,
 'knowledge_gaps': ['Python 함수의 핵심 용어 이해',
  'Python 함수의 가장 기본적인 개념 구분',
  '짧은 예제를 보고 스스로 설명하는 연습'],
 'study_plan': ['5분: Python 함수에서 오늘 꼭 알아야 할 핵심 개념 3개 훑기',
  '15분: 예시 2개를 보며 개념을 직접 설명해 보기',
  '10분: 퀴즈를 풀고 틀린 이유를 한 줄로 정리하기'],
 'lesson_summary': "오늘의 주제는 Python 함수입니다. 목표는 '함수를 직접 정의하고 호출할 수 있다' 달성입니다. 먼저 Python 함수의 정의를 한 문장으로 말할 수 있어야 하고, 그다음에는 대표 예시를 통해 언제 쓰는지 구분할 수 있어야 합니다. 마지막으로 배운 내용을 자신의 말로 다시 설명하면 학습이 더 오래 유지됩니다.",
 'quiz': [{'question': 'Python 함수의 핵심 개념을 한 문장으로 설명해 보세요.',
   'type': 'short_answer'},
  {'question': 'Python 함수가 실제로 사용되는 상황을 한 가지 써 보세요.', 'type': 'application'},
  {'question': '오늘 배운 내용 중 아직 헷갈리는 부분을 한 가지 적어 보세요.', 'type': 'reflection'}],
 'final_feedback': "다음 행동: 5분: Python 함수에서 오늘 꼭 알아야 할 핵심 개념 3개 훑기 후, 퀴즈 3문항을 풀고 헷갈린 개념은 'Python 함수의 핵심 용어 이해' 중심으로 다시 복습하세요."}

In [7]:
print("[학습 공백]")
for gap in result["knowledge_gaps"]:
    print("-", gap)

print("\n[학습 계획]")
for step in result["study_plan"]:
    print("-", step)

print("\n[미니 레슨]")
print(result["lesson_summary"])

print("\n[퀴즈]")
for item in result["quiz"]:
    print(f"- ({item['type']}) {item['question']}")

print("\n[최종 피드백]")
print(result["final_feedback"])


[학습 공백]
- Python 함수의 핵심 용어 이해
- Python 함수의 가장 기본적인 개념 구분
- 짧은 예제를 보고 스스로 설명하는 연습

[학습 계획]
- 5분: Python 함수에서 오늘 꼭 알아야 할 핵심 개념 3개 훑기
- 15분: 예시 2개를 보며 개념을 직접 설명해 보기
- 10분: 퀴즈를 풀고 틀린 이유를 한 줄로 정리하기

[미니 레슨]
오늘의 주제는 Python 함수입니다. 목표는 '함수를 직접 정의하고 호출할 수 있다' 달성입니다. 먼저 Python 함수의 정의를 한 문장으로 말할 수 있어야 하고, 그다음에는 대표 예시를 통해 언제 쓰는지 구분할 수 있어야 합니다. 마지막으로 배운 내용을 자신의 말로 다시 설명하면 학습이 더 오래 유지됩니다.

[퀴즈]
- (short_answer) Python 함수의 핵심 개념을 한 문장으로 설명해 보세요.
- (application) Python 함수가 실제로 사용되는 상황을 한 가지 써 보세요.
- (reflection) 오늘 배운 내용 중 아직 헷갈리는 부분을 한 가지 적어 보세요.

[최종 피드백]
다음 행동: 5분: Python 함수에서 오늘 꼭 알아야 할 핵심 개념 3개 훑기 후, 퀴즈 3문항을 풀고 헷갈린 개념은 'Python 함수의 핵심 용어 이해' 중심으로 다시 복습하세요.


## 확장 아이디어

- `quiz_grader` 노드를 추가해 사용자 답변을 채점할 수 있습니다.
- `memory`를 붙이면 이전 학습 기록을 바탕으로 난이도를 조절할 수 있습니다.
- LLM을 연결하면 과목별로 더 자연스러운 설명과 문제 생성이 가능합니다.